# Phase 1: UDS Dataset Exploration & Variable Inventory

In this notebook, we load and examine the Uniform Data Set (UDS) to understand its structural dimensions, verify column names, and categorize clinical and cognitive variables for subsequent analysis.


In [67]:
import pandas as pd
import numpy as np
from pathlib import Path


data_path = Path("../data/raw/udslong_imputed.csv")
df = pd.read_csv(data_path)


print(f"Dataset Shape -> Rows (Observations): {df.shape[0]}, Columns (Variables): {df.shape[1]}")


Dataset Shape -> Rows (Observations): 1237, Columns (Variables): 180


The dataset contains **180 columns** (variables) across multiple longitudinal observations. These variables encompass demographic information, clinical diagnoses, habits, and neuropsychological test scores (such as MoCA).

# Overview and Categorization of the 180 NACC UDS Variables

1. **Demographics (Variables 1–13):**
   - Core baseline subject profiles such as ID (`NACCID`), biological sex (`SEX`), race, education level (`EDUC` — crucial in Alzheimer's research for cognitive reserve), marital status, and age (`NACCAGE`).

2. **Family History (Variables 14–24):**
   - History of cognitive impairment, Alzheimer's disease (`NACCFADM`), or other dementias in mother, father, and close relatives.

3. **Habits (Variables 25–30):**
   - Prescription medication use, smoking history (duration, packs per day, and age of cessation).

4. **Medical History (Variables 31–74):**
   - History of cardiovascular diseases (myocardial infarction, atrial fibrillation), stroke (`CBSTROKE`), traumatic brain injury (`NACCTBI`), diabetes, hypertension, sleep disorders (apnea, RBD), and psychiatric conditions (depression, anxiety, schizophrenia).

5. **Medication Class (Variables 75–94):**
   - Indices or usage metrics for various drug classes (e.g., aspirin, antihypertensives, statins for hyperlipidemia, diabetes medications, and Parkinson's treatments).

6. **Physical / Vitals (Variables 95–108):**
   - Height, weight, systolic and diastolic blood pressure, heart rate, vision and hearing status, body mass index (`NACCBMI`), and neurological examination results.

7. **NPI-Q (Neuropsychiatric Inventory Questionnaire - Variables 109–133):**
   - Behavioral and psychological symptoms in dementia (e.g., delusions, hallucinations, agitation, depression, apathy, nighttime behavioral disturbances, and appetite changes) along with their respective severity ratings.

8. **GDS (Geriatric Depression Scale - Variables 134–150):**
   - 15 binary (yes/no) items assessing mood, feelings of hope, life satisfaction, and the total depression score (`NACCGDS`).

9. **Functional Activities (Variables 151–160):**
   - Subject's functional independence in complex daily tasks such as financial management (`BILLS`), taxes, shopping, stove usage, meal preparation, keeping track of current events, and traveling.

10. **Cognitive / Diagnostic (Variables 161–180):**
    - Standardized neuropsychological and clinical assessments, including category fluency (animals/vegetables - `ANIMALS`), Trail Making Tests (`TRAILA`, `TRAILB`), MoCA total score (`MOCATOTS`), genetic risk factors (`NACCAPOE` / APOE4 allele), final clinical consensus diagnosis (`NACCUDSD`), and Clinical Dementia Rating (CDR) domain scores (memory, orientation, judgment) along with summary totals (`CDRSUM` / `CDRGLOB`).


# Phase 2: Data Cleaning


### Summary of the Data Cleaning 

This Python script performs standard data cleaning and preprocessing for the NACC UDS medical dataset while strictly preserving all rows (no patient data is dropped before defining a research target).

It executes 5 main steps:

1. **Load Raw Data (`pd.read_csv`)**
   - Automatically checks alternative file paths (`data/raw/` or `../data/raw/`).
   - Reads the CSV file and prints the initial row and column counts.

2. **Categorize Columns Based on NACC Dictionary Rules**
   - Since codes like `8` or `9` have different meanings across columns, it splits them into 3 categories:
     - **a) NPI-Q Severity Columns (`*SEV`):** Code `8` means severity 0 (no symptom), and code `9` means unknown/missing.
     - **b) Cognitive & Test Scores:** Columns like `MOCATOTS` where small numbers (`8`, `9`, etc.) are valid patient scores and must **not** be touched.
     - **c) General Columns:** All other columns where codes like `8`, `88` (Not Applicable) or `9`, `99` (Unknown/Missing) represent errors.

3. **Clean and Standardize Sentinel/Error Codes**
   - Converts 3-digit global error codes (like `777`, `888`, `998`, `999`) across the entire dataset to `NaN`.
   - Applies specific rules to NPI-Q severity columns (mapping `8` to `0` and `9` to `NaN`) and general columns (mapping `8`, `9`, `88`, `99` to `NaN`).

4. **Impute Minor Missing Values (Under 5%)**
   - Checks the missing rate column-by-column.
   - For columns with **between 0% and 5% missing values**:
     - Continuous numeric columns are filled with the **median**.
     - Categorical/discrete/binary columns are filled with the **mode** (most frequent value).
   - Columns with >5% missing data are left untouched, and **no rows are ever dropped**.

5. **Save the Processed Dataset**
   - Automatically creates the output directory (`../data/processed/`) if it doesn't exist.
   - Saves the final cleaned dataset as `udslong_cleaned.csv` and prints the final dimensions.


In [115]:
import os
import numpy as np
import pandas as pd

# 1. Load the raw dataset
input_path = "data/raw/udslong_imputed.csv"
if not os.path.exists(input_path):
    input_path = "../data/raw/udslong_imputed.csv"

df = pd.read_csv(input_path)
print("=" * 60)
print(f"Raw data loaded: {df.shape[0]} rows and {df.shape[1]} columns")
print("=" * 60)

# -------------------------------------------------------------
# 2. Categorize columns based on the NACC data dictionary structure
# -------------------------------------------------------------
id_col = 'NACCID'

# a) NPI-Q severity columns (where code '8' means severity 0)
npi_sev_cols = [c for c in df.columns if c.endswith('SEV')]

# b) Cognitive tests and scores where small numbers are valid scores
cognitive_and_tests = [
    'MOCATOTS', 'ANIMALS', 'VEG', 'TRAILA', 'TRAILARR', 'TRAILALI', 
    'TRAILB', 'TRAILBRR', 'TRAILBLI', 'WAIS', 'BOSTON', 'NACCGDS',
    'CDRSUM', 'EDUC', 'SMOKYRS' 
]


# c) All other columns (where codes 8, 9, 88, 99 represent errors or 'Not Applicable')
other_cols = [
    c for c in df.columns 
    if c != id_col and c not in npi_sev_cols and c not in cognitive_and_tests
]

# -------------------------------------------------------------
# 3. Clean Sentinel/Error codes
# -------------------------------------------------------------

# a) Convert global 3-digit error codes to NaN across the entire dataset
global_sentinels = [777, 777.0, 888, 888.0, 888.8, 998, 998.0, 999, 999.0]
df = df.replace(global_sentinels, np.nan)

# b) Fix NPI-Q severity columns: code 8 = severity 0 | code 9 = NaN
for c in npi_sev_cols:
    if c in df.columns:
        df.loc[df[c] == 8.0, c] = 0.0
        df.loc[df[c] == 9.0, c] = np.nan

# c) Fix general columns (convert codes 8, 9, 88, 99 to NaN)
for c in other_cols:
    if c in df.columns:
        df[c] = df[c].replace([8, 8.0, 9, 9.0, 88, 88.0, 99, 99.0], np.nan)

# -------------------------------------------------------------
# 4. Handle missing values (Impute features with <= 5% missingness )
# -------------------------------------------------------------
feature_cols = [c for c in df.columns if c != id_col]
for col in feature_cols:
    missing_pct = df[col].isnull().mean()
    if 0 < missing_pct <= 0.05:
        if np.issubdtype(df[col].dtype, np.number) and df[col].nunique() > 10:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])


output_dir = '../data/processed' if os.path.exists('../data') else 'data/processed'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'udslong_cleaned.csv')

df.to_csv(output_path, index=False)

print("=" * 60)
print(f"Error code cleaning completed successfully.")
print(f" Final dataset saved to: {output_path}")
print(f" Final dataset dimensions (fully preserved): {df.shape[0]} rows and {df.shape[1]} columns")
print("=" * 60)


Raw data loaded: 1237 rows and 180 columns
Error code cleaning completed successfully.
 Final dataset saved to: ../data/processed/udslong_cleaned.csv
 Final dataset dimensions (fully preserved): 1237 rows and 180 columns


In [96]:
import pandas as pd

pd.set_option('display.max_rows', None)


missing_percentage = (df.isnull().sum() / len(df)) * 100


missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': missing_percentage
})

missing_df = missing_df.sort_values(by='Missing Percentage (%)', ascending=False)

print(missing_df)


          Missing Count  Missing Percentage (%)
NACCOMS            1237              100.000000
NACCFMS            1236               99.919159
NACCAMS            1207               97.574778
HISPOR             1161               93.856103
DIABTYPE           1119               90.460792
HEARWAID           1077               87.065481
NACCAM              832               67.259499
NACCFM              818               66.127728
QUITSMOK            818               66.127728
NACCOM              812               65.642684
ARTHTYPE            538               43.492320
NACCAPOE            278               22.473727
VISWCORR            221               17.865804
NACCAMD             167               13.500404
BPDIAS              136               10.994341
NACCBMI             126               10.185934
NACCFAM             117                9.458367
BPSYS               108                8.730800
TAXES               108                8.730800
HRATE               101                8

In [116]:
missing_summary = df.isnull().sum()
missing_cols = missing_summary[missing_summary > 0]

print(f"Total number of columns that still have missing values:{len(missing_cols)}")
print("-" * 50)
for col, count in missing_cols.items():
    print(f"Column: {col} | Missing Count: {count} | Percentage: {(count / len(df)) * 100:.2f}% | Type: {df[col].dtype}")


Total number of columns that still have missing values:23
--------------------------------------------------
Column: HISPOR | Missing Count: 1161 | Percentage: 93.86% | Type: float64
Column: NACCFAM | Missing Count: 117 | Percentage: 9.46% | Type: float64
Column: NACCAM | Missing Count: 832 | Percentage: 67.26% | Type: float64
Column: NACCAMS | Missing Count: 1207 | Percentage: 97.57% | Type: float64
Column: NACCFM | Missing Count: 818 | Percentage: 66.13% | Type: float64
Column: NACCFMS | Missing Count: 1236 | Percentage: 99.92% | Type: float64
Column: NACCOM | Missing Count: 812 | Percentage: 65.64% | Type: float64
Column: NACCOMS | Missing Count: 1237 | Percentage: 100.00% | Type: float64
Column: QUITSMOK | Missing Count: 818 | Percentage: 66.13% | Type: float64
Column: DIABTYPE | Missing Count: 1119 | Percentage: 90.46% | Type: float64
Column: B12DEF | Missing Count: 69 | Percentage: 5.58% | Type: float64
Column: ARTHTYPE | Missing Count: 538 | Percentage: 43.49% | Type: float64
Co

In [117]:
import os
import pandas as pd

# 1. Load both raw and cleaned datasets
raw_path = 'data/raw/udslong_imputed.csv'
if not os.path.exists(raw_path):
    raw_path = '../data/raw/udslong_imputed.csv'

clean_path = 'data/processed/udslong_cleaned.csv'
if not os.path.exists(clean_path):
    clean_path = '../data/processed/udslong_cleaned.csv'

df_raw = pd.read_csv(raw_path)
df_clean = pd.read_csv(clean_path)

print("=" * 60)
print(" SANITY CHECK & AUDIT REPORT FOR 8 & 9 ERROR CODES")
print("=" * 60)

# 2. Define protected columns (where 8 or 9 are valid clinical/continuous scores)
protected_cols = [
    'MOCATOTS', 'ANIMALS', 'VEG', 'TRAILA', 'TRAILARR', 'TRAILALI', 
    'TRAILB', 'TRAILBRR', 'TRAILBLI', 'WAIS', 'BOSTON', 'NACCGDS',
    'CDRSUM', 'EDUC', 'SMOKYRS'
]

print("\n--- A) Checking Protected Columns (Must show 0 changes in 8s/9s) ---")
for col in protected_cols:
    if col in df_raw.columns and col in df_clean.columns:
        raw_eights = (df_raw[col] == 8).sum() + (df_raw[col] == 8.0).sum()
        clean_eights = (df_clean[col] == 8).sum() + (df_clean[col] == 8.0).sum()
        
        if raw_eights != clean_eights:
            print(f"Warning in {col}: Count of 8s changed! (Raw: {raw_eights}, Clean: {clean_eights})")
        else:
            print(f"{col}: Successfully protected (Count of 8s unchanged: {int(raw_eights)})")

# 3. Check general columns (where error codes should be successfully converted to NaN)
print("\n--- B) Sample Audit of General Columns (Error codes converted to NaN) ---")
sample_cols = ['HISPOR', 'DIABETES', 'HYPERTEN', 'BILLS', 'SHOPPING']
for col in sample_cols:
    if col in df_raw.columns and col in df_clean.columns:
        raw_nans = df_raw[col].isnull().sum()
        clean_nans = df_clean[col].isnull().sum()
        diff = clean_nans - raw_nans
        print(f"Column {col}: Missing values increased from {raw_nans} to {clean_nans} (+{diff} converted to NaN)")

print("=" * 60)
print(f"Audit completed successfully. Total rows preserved: {df_clean.shape[0]}")
print("=" * 60)


 SANITY CHECK & AUDIT REPORT FOR 8 & 9 ERROR CODES

--- A) Checking Protected Columns (Must show 0 changes in 8s/9s) ---
MOCATOTS: Successfully protected (Count of 8s unchanged: 12)
ANIMALS: Successfully protected (Count of 8s unchanged: 22)
VEG: Successfully protected (Count of 8s unchanged: 108)
TRAILA: Successfully protected (Count of 8s unchanged: 0)
TRAILARR: Successfully protected (Count of 8s unchanged: 0)
TRAILALI: Successfully protected (Count of 8s unchanged: 2)
TRAILB: Successfully protected (Count of 8s unchanged: 0)
TRAILBRR: Successfully protected (Count of 8s unchanged: 6)
TRAILBLI: Successfully protected (Count of 8s unchanged: 6)
NACCGDS: Successfully protected (Count of 8s unchanged: 26)
CDRSUM: Successfully protected (Count of 8s unchanged: 18)
EDUC: Successfully protected (Count of 8s unchanged: 4)
SMOKYRS: Successfully protected (Count of 8s unchanged: 10)

--- B) Sample Audit of General Columns (Error codes converted to NaN) ---
Column HISPOR: Missing values incre

### 📝 Data Retention Rationale: Handling Missing Values in the Remaining 25 Variables

During the data cleaning and preprocessing phase of the NACC UDS longitudinal dataset, **25 variables** were identified as having missing values (ranging from ~5% to nearly 100% missingness). 

Instead of imputing these values with arbitrary statistics (like mean/median/mode) or dropping the affected rows/columns, we decided to **retain all 1,237 rows and keep these variables as `NaN` (Missing)** for the following strict statistical and clinical reasons:

#### 1. Conditional and Skip-Pattern Nature of Clinical Questionnaires
A significant portion of these variables (e.g., `HISPOR`, `DIABTYPE`, `QUITSMOK`, `ARTHTYPE`, `HEARWAID`) are **conditional or follow-up items** in clinical interviews. For instance:
- `DIABTYPE` (Type of diabetes) is only asked if a patient reports having diabetes (`DIABETES = 1`). 
- For non-diabetic patients, this field is naturally skipped and left blank. Imputing or deleting based on this would introduce severe structural bias.

#### 2. Preservation of Clinical and Functional Integrity
Variables such as `BILLS` (financial management), `TAXES`, or `B12DEF` (vitamin B12 deficiency) measure specific cognitive, functional, or medical states. 
- Using generic imputation methods (like median or mode) on functional/medical scores distorts the clinical reality of the patient.
- Imputing missing values here risks masking actual cognitive impairments or physical conditions.

#### 3. Statistical Rigor and Avoiding Variance Distortion
- In pure statistical analysis, standard statistical procedures (such as regression models or hypothesis tests) handle missing data dynamically (using available case analysis) without requiring artificial value substitution.
- Imputing continuous variables like `WEIGHT` or `NACCBMI` with the median artificially reduces variance and distorts true underlying correlations.

#### 4. Absence of a Defined Target Variable
Since the primary target variable (dependent variable) for our specific statistical hypothesis has not yet been finalized, **no rows or variables are prematurely discarded or altered**. Preserving the raw structure ensures maximum flexibility for future hypothesis-driven modeling.

**Conclusion:** 
All 1,237 rows and 180 columns are safely maintained. Missing values in these 25 variables are kept as `NaN` to reflect true clinical workflows and ensure unbiased future statistical testing.


In [99]:
import os
import pandas as pd

# 1. Load the MRI dataset with robust path checking
input_path = "data/raw/mriscanclean.csv"
if not os.path.exists(input_path):
    input_path = "../data/raw/mriscanclean.csv"

if not os.path.exists(input_path):
    # Fallback to absolute or current workspace locations if needed
    input_path = "mriscanclean.csv"

df_mri = pd.read_csv(input_path)

print("=" * 60)
print(f" MRI Dataset Loaded Successfully!")
print(f" Shape: {df_mri.shape[0]} rows and {df_mri.shape[1]} columns")
print("=" * 60)

# 2. Check for NACCID presence
if 'NACCID' in df_mri.columns:
    print(f" 'NACCID' column found.")
    print(f"   - Unique patients (NACCIDs): {df_mri['NACCID'].nunique()}")
    print(f"   - Missing NACCIDs: {df_mri['NACCID'].isna().sum()}")
    print(f"   - Duplicate rows per NACCID: {df_mri.duplicated(subset=['NACCID']).sum()}")
else:
    print(" Warning: 'NACCID' column not found in this CSV! Please check column names.")

print("-" * 60)
print(" Missing Values Summary (Top columns with missing data):")
missing_counts = df_mri.isnull().sum()
missing_percentages = (missing_counts / len(df_mri)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_percentages})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Percentage (%)', ascending=False)

if not missing_df.empty:
    print(missing_df.head(15))
else:
    print(" No missing values found in this dataset.")

print("=" * 60)
print(" First 5 Rows Preview:")
display(df_mri.head())


 MRI Dataset Loaded Successfully!
 Shape: 1237 rows and 126 columns
 'NACCID' column found.
   - Unique patients (NACCIDs): 1237
   - Missing NACCIDs: 0
   - Duplicate rows per NACCID: 0
------------------------------------------------------------
 Missing Values Summary (Top columns with missing data):
 No missing values found in this dataset.
 First 5 Rows Preview:


,NACCID,LH_CAUDALANTERIORCINGULATE_GVOL,RH_CAUDALANTERIORCINGULATE_GVOL,LH_CAUDALANTERIORCINGULATE_AVGTH,RH_CAUDALANTERIORCINGULATE_AVGTH,LH_CAUDALMIDDLEFRONTAL_GVOL,RH_CAUDALMIDDLEFRONTAL_GVOL,LH_CAUDALMIDDLEFRONTAL_AVGTH,RH_CAUDALMIDDLEFRONTAL_AVGTH,LH_CUNEUS_GVOL,...,RH_SUPRAMARGINAL_AVGTH,LH_TRANSVERSETEMPORAL_GVOL,RH_TRANSVERSETEMPORAL_GVOL,LH_TRANSVERSETEMPORAL_AVGTH,RH_TRANSVERSETEMPORAL_AVGTH,LH_INSULA_GVOL,RH_INSULA_GVOL,LH_INSULA_AVGTH,RH_INSULA_AVGTH,NACCUDSD
0,NACC359836,3462,2257,2.328,2.278,6767,5857,2.274,2.303,5726,...,2.330,1028,786,2.060,2.075,5785,6224,3.045,3.129,1
1,NACC990210,3131,2528,2.553,2.292,6898,6880,2.478,2.415,4512,...,2.383,1430,783,2.193,2.479,6075,6200,2.999,2.846,1
2,NACC074744,2463,1618,2.060,1.942,7176,5770,2.254,2.268,4240,...,2.158,1299,1036,2.204,2.202,5912,6344,2.655,2.847,3
3,NACC083525,2980,2302,2.405,2.230,6915,6303,2.620,2.618,4790,...,2.599,1061,846,2.445,2.551,6226,6050,3.117,3.026,1
4,NACC092141,3000,1703,2.404,2.140,6655,6556,2.454,2.406,4075,...,2.427,1263,784,2.543,2.591,5582,5743,3.089,2.853,1


In [100]:
from pathlib import Path
import re
import pandas as pd

# ------------------------------------------------------------
# 1. Find the files
# The code searches the current folder and its parent folders.
# ------------------------------------------------------------
roots = [Path.cwd(), *Path.cwd().parents]

def find_file(relative_paths):
    for root in roots:
        for relative_path in relative_paths:
            candidate = root / relative_path
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        "Could not find any of these files:\n"
        + "\n".join(relative_paths)
        + f"\n\nCurrent working directory: {Path.cwd()}"
    )

mri_path = find_file([
    "data/raw/mriscanclean.csv",
    "mriscanclean.csv",
])

clinical_path = find_file([
    "data/processed/udslong_cleaned.csv",
    "data/raw/udslong_imputed.csv",
])

print("MRI file:", mri_path)
print("Clinical file:", clinical_path)

# Read NACCID as text so its format is preserved
df_mri = pd.read_csv(mri_path, dtype={"NACCID": "string"})
df_clinical = pd.read_csv(clinical_path, dtype={"NACCID": "string"})


# ------------------------------------------------------------
# 2. NACCUDSD distribution
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("1) NACCUDSD VALUE DISTRIBUTION")
print("=" * 70)

if "NACCUDSD" in df_mri.columns:
    counts = df_mri["NACCUDSD"].value_counts(dropna=False).sort_index()
    distribution = pd.DataFrame({
        "Count": counts,
        "Percentage": (counts / len(df_mri) * 100).round(2)
    })
    print(distribution)
else:
    print("NACCUDSD was not found in the MRI file.")

print(
    "\nImportant: Confirm the meaning of each NACCUDSD code "
    "using the relevant NACC data dictionary before assigning labels."
)


# ------------------------------------------------------------
# 3. Extract MRI regions and check hemisphere/measurement coverage
# Expected column pattern: LH_REGION_GVOL, RH_REGION_AVGTH, etc.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2) MRI MEASUREMENTS AND BRAIN REGIONS")
print("=" * 70)

pattern = re.compile(r"^(LH|RH)_(.+)_(GVOL|AVGTH)$")
records = []

for column in df_mri.columns:
    match = pattern.match(column)
    if match:
        hemisphere, region, measure = match.groups()
        records.append({
            "Column": column,
            "Hemisphere": hemisphere,
            "Region": region,
            "Measure": measure
        })

measurements = pd.DataFrame(records)

if measurements.empty:
    print("No columns matched the expected MRI naming pattern.")
else:
    regions = sorted(measurements["Region"].unique())

    print(f"Number of MRI measurement columns: {len(measurements)}")
    print(f"Number of distinct brain regions: {len(regions)}")
    print("\nNumber of columns by measure and hemisphere:")
    print(pd.crosstab(measurements["Measure"], measurements["Hemisphere"]))

    print("\nBrain regions:")
    for region in regions:
        print("-", region)

    # Check whether each region has all four expected measurements:
    # LH_GVOL, RH_GVOL, LH_AVGTH, RH_AVGTH
    expected = {
        (hemisphere, measure)
        for hemisphere in ["LH", "RH"]
        for measure in ["GVOL", "AVGTH"]
    }

    coverage = (
        measurements.groupby("Region")
        .apply(lambda group: expected - set(zip(group["Hemisphere"], group["Measure"])))
    )

    incomplete_regions = {
        region: missing
        for region, missing in coverage.items()
        if missing
    }

    if incomplete_regions:
        print("\nRegions missing one or more expected measurements:")
        for region, missing in incomplete_regions.items():
            print(f"- {region}: missing {sorted(missing)}")
    else:
        print("\nAll regions have the expected left/right GVOL and AVGTH columns.")

    nonmatching_columns = [
        col for col in df_mri.columns
        if col not in measurements["Column"].tolist()
        and col not in ["NACCID", "NACCUDSD"]
    ]
    print("\nOther columns not matching the MRI naming pattern:")
    print(nonmatching_columns if nonmatching_columns else "None")


# ------------------------------------------------------------
# 4. Compare NACCID values between MRI and clinical datasets
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("3) NACCID MATCH BETWEEN MRI AND CLINICAL DATA")
print("=" * 70)

if "NACCID" not in df_mri.columns or "NACCID" not in df_clinical.columns:
    print("NACCID is missing from one or both datasets.")
else:
    mri_ids = df_mri["NACCID"].astype("string").str.strip()
    clinical_ids = df_clinical["NACCID"].astype("string").str.strip()

    print(f"MRI rows: {len(df_mri):,}")
    print(f"Clinical rows: {len(df_clinical):,}")
    print(f"MRI missing NACCID: {mri_ids.isna().sum():,}")
    print(f"Clinical missing NACCID: {clinical_ids.isna().sum():,}")

    mri_nonmissing = mri_ids.dropna()
    clinical_nonmissing = clinical_ids.dropna()

    print(
        "Duplicate NACCID rows in MRI:",
        mri_nonmissing.duplicated().sum()
    )
    print(
        "Duplicate NACCID rows in clinical data:",
        clinical_nonmissing.duplicated().sum()
    )

    mri_id_set = set(mri_nonmissing)
    clinical_id_set = set(clinical_nonmissing)

    shared_ids = mri_id_set & clinical_id_set
    only_in_mri = mri_id_set - clinical_id_set
    only_in_clinical = clinical_id_set - mri_id_set

    print(f"\nUnique MRI IDs: {len(mri_id_set):,}")
    print(f"Unique clinical IDs: {len(clinical_id_set):,}")
    print(f"IDs present in both datasets: {len(shared_ids):,}")
    print(f"IDs only in MRI: {len(only_in_mri):,}")
    print(f"IDs only in clinical data: {len(only_in_clinical):,}")

    print("\nExample IDs only in MRI:", sorted(only_in_mri)[:10])
    print("Example IDs only in clinical data:", sorted(only_in_clinical)[:10])


# ------------------------------------------------------------
# 5. Screen MRI measures for non-positive values and IQR outliers
# This is a screening step only; do not automatically delete values.
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("4) NON-POSITIVE VALUES AND POSSIBLE IQR OUTLIERS")
print("=" * 70)

if not measurements.empty:
    mri_columns = measurements["Column"].tolist()
    outlier_rows = []
    invalid_rows = []

    for column in mri_columns:
        values = pd.to_numeric(df_mri[column], errors="coerce")
        nonnumeric_count = (df_mri[column].notna() & values.isna()).sum()
        nonpositive_count = (values.notna() & (values <= 0)).sum()

        if nonnumeric_count > 0 or nonpositive_count > 0:
            invalid_rows.append({
                "Column": column,
                "Non-numeric values": int(nonnumeric_count),
                "Values <= 0": int(nonpositive_count)
            })

        valid_values = values.dropna()
        if len(valid_values) >= 4:
            q1 = valid_values.quantile(0.25)
            q3 = valid_values.quantile(0.75)
            iqr = q3 - q1

            lower_fence = q1 - 1.5 * iqr
            upper_fence = q3 + 1.5 * iqr
            outlier_mask = (valid_values < lower_fence) | (valid_values > upper_fence)

            outlier_rows.append({
                "Column": column,
                "IQR outlier count": int(outlier_mask.sum()),
                "Lower IQR fence": round(lower_fence, 3),
                "Upper IQR fence": round(upper_fence, 3)
            })

    print("Columns with non-numeric or non-positive measurement values:")
    if invalid_rows:
        print(pd.DataFrame(invalid_rows).to_string(index=False))
    else:
        print("None found.")

    outlier_summary = pd.DataFrame(outlier_rows)
    outlier_summary = outlier_summary.sort_values(
        "IQR outlier count", ascending=False
    )

    print("\nColumns with the most possible IQR outliers:")
    print(outlier_summary.head(20).to_string(index=False))

    print(
        "\nReminder: IQR flags are only a screening tool. "
        "An IQR-flagged MRI value is not automatically an error "
        "and should not be deleted without checking the data and methodology."
    )
else:
    print("No MRI measurement columns were identified for screening.")


MRI file: /home/parnia/nacc_project/data/raw/mriscanclean.csv
Clinical file: /home/parnia/nacc_project/data/processed/udslong_cleaned.csv

1) NACCUDSD VALUE DISTRIBUTION
          Count  Percentage
NACCUDSD                   
1           847       68.47
2             7        0.57
3           252       20.37
4           131       10.59

Important: Confirm the meaning of each NACCUDSD code using the relevant NACC data dictionary before assigning labels.

2) MRI MEASUREMENTS AND BRAIN REGIONS
Number of MRI measurement columns: 124
Number of distinct brain regions: 31

Number of columns by measure and hemisphere:
Hemisphere  LH  RH
Measure           
AVGTH       31  31
GVOL        31  31

Brain regions:
- CAUDALANTERIORCINGULATE
- CAUDALMIDDLEFRONTAL
- CUNEUS
- ENTORHINAL
- FUSIFORM
- INFERIORPARIETAL
- INFERIORTEMPORAL
- INSULA
- ISTHMUSCINGULATE
- LATERALOCCIPITAL
- LATERALORBITOFRONTAL
- LINGUAL
- MEDIALORBITOFRONTAL
- MIDDLETEMPORAL
- PARACENTRAL
- PARAHIPPOCAMPAL
- PARSOPERCULARIS
- 

In [101]:
# Check non-positive rows in the MRI dataset
cols_to_check = ['LH_ENTORHINAL_GVOL', 'LH_POSTERIORCINGULATE_GVOL']
bad_rows = df_mri[(df_mri['LH_ENTORHINAL_GVOL'] <= 0) | (df_mri['LH_POSTERIORCINGULATE_GVOL'] <= 0)]

print(f"Number of problematic rows found: {len(bad_rows)}")
display(bad_rows[['NACCID', 'LH_ENTORHINAL_GVOL', 'RH_ENTORHINAL_GVOL', 
                  'LH_POSTERIORCINGULATE_GVOL', 'RH_POSTERIORCINGULATE_GVOL', 'NACCUDSD']])


Number of problematic rows found: 2


,NACCID,LH_ENTORHINAL_GVOL,RH_ENTORHINAL_GVOL,LH_POSTERIORCINGULATE_GVOL,RH_POSTERIORCINGULATE_GVOL,NACCUDSD
470,NACC684612,211,348,0,343,4
518,NACC931831,0,350,610,636,1


In [102]:
import numpy as np
import pandas as pd

# 1. بارگذاری مجدد فایل MRI
df_mri_clean = df_mri.copy()

# لیست ستون‌های سنجش MRI (۱۲۴ ستون کورتیکال)
mri_feature_cols = [c for c in df_mri_clean.columns if c not in ['NACCID', 'NACCUDSD']]

# 2. بررسی تعداد مقادیر 0 قبل از اصلاح
zeros_before = (df_mri_clean[mri_feature_cols] <= 0).sum().sum()
print(f"تعداد مقادیر نامعتبر (<= 0) قبل از پاکسازی: {zeros_before}")

# 3. تبدیل مقادیر <= 0 به NaN
for col in mri_feature_cols:
    df_mri_clean.loc[df_mri_clean[col] <= 0, col] = np.nan

# 4. بررسی پس از اصلاح
zeros_after = (df_mri_clean[mri_feature_cols] <= 0).sum().sum()
nans_created = df_mri_clean[mri_feature_cols].isna().sum().sum()
print(f"تعداد مقادیر نامعتبر بعد از پاکسازی: {zeros_after}")
print(f"تعداد کل مقادیر NaN در فیچرهای MRI: {nans_created}")

# 5. مشاهده مجدد همان دو سطر برای اطمینان
print("\nوضعیت همان دو سطر پس از تبدیل:")
display(df_mri_clean.loc[[470, 518], ['NACCID', 'LH_ENTORHINAL_GVOL', 'RH_ENTORHINAL_GVOL', 
                                      'LH_POSTERIORCINGULATE_GVOL', 'RH_POSTERIORCINGULATE_GVOL', 'NACCUDSD']])

# 6. ذخیره در پوشه data/processed
output_path = "/home/parnia/nacc_project/data/processed/mriscan_cleaned.csv"
df_mri_clean.to_csv(output_path, index=False)
print(f"\n✅ فایل تمیز MRI با موفقیت ذخیره شد در:\n{output_path}")


تعداد مقادیر نامعتبر (<= 0) قبل از پاکسازی: 2
تعداد مقادیر نامعتبر بعد از پاکسازی: 0
تعداد کل مقادیر NaN در فیچرهای MRI: 2

وضعیت همان دو سطر پس از تبدیل:


,NACCID,LH_ENTORHINAL_GVOL,RH_ENTORHINAL_GVOL,LH_POSTERIORCINGULATE_GVOL,RH_POSTERIORCINGULATE_GVOL,NACCUDSD
470,NACC684612,211.0,348.0,NaN,343.0,4
518,NACC931831,NaN,350.0,610.0,636.0,1



✅ فایل تمیز MRI با موفقیت ذخیره شد در:
/home/parnia/nacc_project/data/processed/mriscan_cleaned.csv


## 2. Unit of Observation and Data Structure

A row is the basic observational unit in a dataset. We used `NACCID` to
determine whether each participant contributes one row or multiple rows.

### Unit of observation

- **Clinical UDS dataset:** Each row represents one participant (`NACCID`) and
  the clinical information recorded for that participant in the selected
  assessment record.
- **MRI dataset:** Each row represents one participant (`NACCID`) and the MRI
  measurements recorded for that participant.

In the current cleaned files, each participant appears once in each dataset.
Therefore, each file contains one record per participant. The MRI and UDS
records are linked using `NACCID`.

### Data structure

The current files have a **one-row-per-participant, cross-sectional structure**.
No participant contributes multiple rows in either file, so the current
analysis files do not contain repeated observations per participant.


                 Current analysis files

     UDS clinical file                    MRI file
     1 row per NACCID                     1 row per NACCID
             │                                   │
             └──────── linked by NACCID ─────────┘
                         1:1 match

     Participant A: one UDS row + one MRI row
     Participant B: one UDS row + one MRI row
     ...


In [106]:

import pandas as pd

def check_structure(df, name):
    print("=" * 60)
    print(f"Dataset: {name}")
    print(f"Total rows: {len(df):,}")

    if "NACCID" not in df.columns:
        print("ERROR: NACCID column was not found.")
        return

    missing_ids = df["NACCID"].isna().sum()
    counts = df["NACCID"].dropna().value_counts()

    print(f"Unique NACCIDs: {df['NACCID'].nunique():,}")
    print(f"Missing NACCIDs: {missing_ids}")

    if counts.empty:
        print("No non-missing NACCIDs are available to assess.")
    else:
        print(f"Minimum rows per participant: {counts.min()}")
        print(f"Maximum rows per participant: {counts.max()}")

        if counts.max() == 1 and missing_ids == 0:
            print("Result: One row per participant; cross-sectional structure.")
        else:
            repeated_people = counts[counts > 1]
            print("Result: Repeated records and/or missing participant IDs were found.")

            if not repeated_people.empty:
                print(f"Participants with multiple rows: {len(repeated_people)}")
                print("Distribution of rows per participant:")
                print(counts.value_counts().sort_index().to_string())

    print("-" * 60)


clinical_df = pd.read_csv(
    "../data/processed/udslong_cleaned.csv",
    dtype={"NACCID": "string"}
)

mri_df = pd.read_csv(
    "../data/processed/mriscan_cleaned.csv",
    dtype={"NACCID": "string"}
)

check_structure(clinical_df, "Clinical UDS")
check_structure(mri_df, "MRI")



Dataset: Clinical UDS
Total rows: 1,237
Unique NACCIDs: 1,237
Missing NACCIDs: 0
Minimum rows per participant: 1
Maximum rows per participant: 1
Result: One row per participant; cross-sectional structure.
------------------------------------------------------------
Dataset: MRI
Total rows: 1,237
Unique NACCIDs: 1,237
Missing NACCIDs: 0
Minimum rows per participant: 1
Maximum rows per participant: 1
Result: One row per participant; cross-sectional structure.
------------------------------------------------------------



# 1. Overview and Variable Mapping of the UDS and MRI Datasets

This research relies on two core datasets that are linked 1:1 via a unique subject identifier (`NACCID`). Combining these sources provides a comprehensive profile of both clinical status and structural brain morphometry.

---

## A) UDS Clinical Dataset: Variable Categorization (180 Variables)

The clinical dataset captures longitudinal and cross-sectional non-imaging assessments, detailing participants' demographics, general health, daily functioning, and cognitive performance. These 180 variables are organized into 10 main domains:

1. **Demographics (Variables 1–13):**
   - Baseline data including subject ID (`NACCID`), biological sex (`SEX`), race, education level (`EDUC` — vital for cognitive reserve assessment), marital status, and age (`NACCAGE`).

2. **Family History (Variables 14–24):**
   - History of cognitive impairment, Alzheimer's disease (`NACCFADM`), or other dementias in biological parents and first-degree relatives.

3. **Habits & Lifestyle (Variables 25–30):**
   - Prescription drug usage and smoking patterns (duration, packs per year/day, and cessation age).

4. **Medical History & Comorbidities (Variables 31–74):**
   - Systemic health conditions such as cardiovascular disease, stroke history (`CBSTROKE`), traumatic brain injury (`NACCTBI`), diabetes, hypertension, sleep disorders (e.g., REM sleep behavior disorder), and psychiatric history.

5. **Medication Classes (Variables 75–94):**
   - Structured tracking of ongoing pharmacological treatments (antihypertensives, cholesterol-lowering statins, antidiabetic drugs, and Parkinson's therapeutics).

6. **Physical / Vitals & Neurological Examinations (Variables 95–108):**
   - Height, weight, systolic and diastolic blood pressure, resting heart rate, sensory impairments, and body mass index (`NACCBMI`).

7. **Neuropsychiatric Inventory Questionnaire (NPI-Q; Variables 109–133):**
   - Behavioral and psychological symptoms of dementia (such as delusions, hallucinations, agitation, apathy, and nighttime behavioral disturbances) along with their respective severity scores.

8. **Geriatric Depression Scale (GDS; Variables 134–150):**
   - 15 binary (yes/no) screening items evaluating depressive symptoms, life satisfaction, feelings of helplessness, and the aggregated depression score (`NACCGDS`).

9. **Functional Activities Questionnaire (FAQ; Variables 151–160):**
   - Instrumental activities of daily living (IADLs) evaluating real-world independence, such as managing finances (`BILLS`), grocery shopping, cooking, stove safety, and independent traveling.

10. **Cognitive, Neuropsychological & Diagnostic Profile (Variables 161–180):**
    - Specialized cognitive tests including category verbal fluency (`ANIMALS`, `VEG`), processing speed and executive functions (`TRAILA`, `TRAILB`), the MoCA test, genetic risk markers (`NACCAPOE`), CDR metrics (box and global scores), and the final clinical consensus diagnosis (`NACCUDSD`).

---

## B) MRI Neuroimaging Dataset: Structural Brain Features (126 Features)

This dataset includes volumetric and morphometric features extracted from the brain MRI scans of 1,237 participants, tracking structural changes across 31 bilateral regions of interest (ROIs):

1. **Identifiers & Clinical Consensus (2 variables):**
   - Includes `NACCID` for merging with clinical data and `NACCUDSD` representing the participant's diagnostic status (ranging from normal cognition to full dementia).

2. **Cortical Gray Matter Volume (`*_GVOL` — 62 variables):**
   - Regional volume in cubic millimeters ($mm^3$) across various cortical structures; reductions here indicate atrophy and neuronal loss (key regions for Alzheimer's include the entorhinal cortex, hippocampus, and posterior cingulate).

3. **Cortical Thickness (`*_AVGTH` — 62 variables):**
   - The mean distance ($mm$) between the pial surface and the gray-white matter boundary. This is a highly sensitive biomarker that often captures cortical thinning prior to gross volumetric shrinkage.

4. **Hemispheric Lateralization (`LH_` vs. `RH_`):**
   - All anatomical regions are measured independently for the **Left Hemisphere (`LH_`)** and **Right Hemisphere (`RH_`)**, allowing for the comparison of structural symmetry and disease progression patterns across hemispheres.

---

## C) Synthesis and Data Integration

- **Clinical Data (UDS)** describes the phenotypic expression of the individual (e.g., age, risk factors, functional independence, and cognitive test scores).
- **Neuroimaging Data (MRI)** quantifies the biological and physical changes in the brain (e.g., regional atrophy and cortical thickness).

By merging these two domains via the `NACCID` key, we can analyze how structural brain degradation correlates with real-world functional decline and cognitive impairment in patients.

In [110]:
import pandas as pd

# Load datasets
uds = pd.read_csv("../data/processed/udslong_cleaned.csv")
mri = pd.read_csv("../data/processed/mriscan_cleaned.csv")

print("UDS shape:", uds.shape)
print("MRI shape:", mri.shape)


UDS shape: (1237, 180)
MRI shape: (1237, 126)


In [ ]:

mri_summary = pd.DataFrame({
    "Category": [
        "Identifier / linkage",
        "Gray matter volume",
        "Cortical thickness",
        "Left hemisphere features",
        "Right hemisphere features"
    ],
    "Definition": [
        "Variables used to identify and merge participants across files",
        "Regional structural brain volume measures",
        "Average cortical thickness measures for anatomical regions",
        "MRI features measured in the left hemisphere",
        "MRI features measured in the right hemisphere"
    ],
    "Count": [
        len(id_vars),
        len(gvol_vars),
        len(avgth_vars),
        len(lh_vars),
        len(rh_vars)
    ]
})

mri_summary


,Category,Definition,Count
0,Identifier / linkage,Variables used to identify and merge participa...,2
1,Gray matter volume,Regional structural brain volume measures,62
2,Cortical thickness,Average cortical thickness measures for anatom...,62
3,Left hemisphere features,MRI features measured in the left hemisphere,62
4,Right hemisphere features,MRI features measured in the right hemisphere,62


## C.2. Participant and Visit Structure Analysis

### A) Empirical Findings from the Analytical Datasets
We evaluated the participant-to-visit structure across both the clinical (`udslong_cleaned.csv`) and neuroimaging (`mriscan_cleaned.csv`) datasets:

1. **Unique Participants vs. Total Rows:**
   - **Clinical Dataset:** 1,237 total rows, 1,237 non-missing `NACCID` values, and **1,237 unique participants**.
   - **MRI Dataset:** 1,237 total rows, 1,237 non-missing `NACCID` values, and **1,237 unique participants**.
2. **Visit Distribution:**
   - In both files, **100% of participants have exactly one row** ($n = 1,237, 100\%$). No participant appears more than once ($0$ participants with $>1$ row).
   - Inspection of the clinical feature space confirms no longitudinal visit identifiers (such as `NACCVISIT`, `VISCODE`, or `EXAMDATE`) were retained in this processed snapshot.

### B) Scientific Interpretation
Although the source NACC cohort is inherently longitudinal, the curated dataset provided for this track has already been pre-filtered into a **cross-sectional, matched cohort**. Each participant's clinical profile represents a single, synchronized observation aligned with their structural MRI scan.

### C) Primary-Observation Selection Rule (Independence Guarantee)
In biomedical and statistical inference, treating repeated measures from the same subject as independent observations leads to pseudo-replication, deflated standard errors, and inflated Type I error rates. 

To guarantee the **independence of observations** assumption for downstream statistical analyses (e.g., hypothesis testing, regression models, classification):
1. **Pre-selected Linked Observation:** Because the working dataset has strictly one row per `NACCID`, each participant enters subsequent analyses as exactly **one independent statistical observation ($N = 1,237$)**.
2. **Theoretical Rule for Raw Longitudinal Data:** If the raw longitudinal NACC files were to be re-extracted, our primary selection rule would be to select the **index visit concurrent with the MRI acquisition** (within an acceptable temporal window, e.g., $\pm 90$ days), rather than selecting visits based on clinical outcome or extreme cognitive scores (which would introduce selection bias).
3. **Repeated Measures Handling:** Should multiple longitudinal visits be incorporated in future phases, they will not be treated as independent cases; instead, mixed-effects models (hierarchical linear modeling) or Generalized Estimating Equations (GEE) with patient-level clustering will be applied.


In [ ]:
import pandas as pd


uds_clean = pd.read_csv("../data/processed/udslong_cleaned.csv")
mri_clean = pd.read_csv("../data/processed/mriscan_cleaned.csv")

def get_structure_metrics(df, dataset_name):
    total_rows = len(df)
    unique_ids = df['NACCID'].nunique()
    counts_per_id = df.groupby('NACCID').size()
    
    return {
        "Dataset": dataset_name,
        "Total Rows": total_rows,
        "Unique Participants (NACCID)": unique_ids,
        "Participants with 1 Row": (counts_per_id == 1).sum(),
        "Participants with >1 Rows": (counts_per_id > 1).sum(),
        "Max Rows per Participant": counts_per_id.max(),
        "Data Structure": "Cross-sectional (1:1)" if total_rows == unique_ids else "Longitudinal / Clustered"
    }

structure_summary = pd.DataFrame([
    get_structure_metrics(uds_clean, "UDS Clinical Dataset"),
    get_structure_metrics(mri_clean, "MRI Imaging Dataset")
])

print("=" * 70)
print("PARTICIPANT & VISIT STRUCTURE REPORT (TASK C.2)")
print("=" * 70)
display(structure_summary)

assert structure_summary["Total Rows"].equals(structure_summary["Unique Participants (NACCID)"]), \
    "Warning: Multiple rows detected per participant!"
print("\n[CONFIRMED]: Every record represents an independent unique participant (No repeated measures).")


PARTICIPANT & VISIT STRUCTURE REPORT (TASK C.2)


,Dataset,Total Rows,Unique Participants (NACCID),Participants with 1 Row,Participants with >1 Rows,Max Rows per Participant,Data Structure
0,UDS Clinical Dataset,1237,1237,1237,0,1,Cross-sectional (1:1)
1,MRI Imaging Dataset,1237,1237,1237,0,1,Cross-sectional (1:1)



[CONFIRMED]: Every record represents an independent unique participant (No repeated measures).


In [ ]:
import pandas as pd

uds_path = "../data/processed/udslong_cleaned.csv"
mri_path = "../data/processed/mriscan_cleaned.csv"

uds = pd.read_csv(uds_path)
mri = pd.read_csv(mri_path)

key = "NACCID"


for name, df in [("UDS", uds), ("MRI", mri)]:
    if key not in df.columns:
        raise KeyError(f"{key!r} is missing from the {name} dataset.")
    if df[key].isna().any():
        raise ValueError(f"{name} contains missing {key} values.")
    if df[key].duplicated().any():
        raise ValueError(f"{name} contains duplicate {key} values.")

uds_ids = set(uds[key])
mri_ids = set(mri[key])

common_ids = uds_ids & mri_ids
uds_only_ids = uds_ids - mri_ids
mri_only_ids = mri_ids - uds_ids

merged = uds.merge(
    mri,
    on=key,
    how="inner",
    suffixes=("_UDS", "_MRI"),
    validate="one_to_one"
)

print("=== UDS-MRI LINKAGE REPORT ===")
print(f"UDS unique participants:       {len(uds_ids):,}")
print(f"MRI unique participants:       {len(mri_ids):,}")
print(f"Participants in both files:    {len(common_ids):,}")
print(f"UDS-only participants:         {len(uds_only_ids):,}")
print(f"MRI-only participants:         {len(mri_only_ids):,}")
print(f"Final inner-joined sample:     {merged[key].nunique():,}")
print(f"Rows in merged dataset:        {len(merged):,}")

assert len(merged) == len(common_ids)
assert merged[key].is_unique


=== UDS-MRI LINKAGE REPORT ===
UDS unique participants:       1,237
MRI unique participants:       1,237
Participants in both files:    1,237
UDS-only participants:         0
MRI-only participants:         0
Final inner-joined sample:     1,237
Rows in merged dataset:        1,237


## C.3. Linking the UDS and MRI Datasets

The UDS clinical and MRI datasets were linked using the participant identifier `NACCID`. Before merging, we checked that the identifier was present and non-missing in both files, and counted unique participants in each dataset.

The UDS file contained **1,237 unique participants**, and the MRI file contained **1,237 unique participants**. The two files had **1,237 participants in common**. There were **0 UDS-only participants** and **0 MRI-only participants**.

A one-to-one inner join on `NACCID` therefore produced a final linked sample of **1,237 participants**. No participants were lost during linkage. The merge validation confirmed that each participant had at most one record in each file.
